In [4]:
import numpy as np
import pandas as pd

print("NumPy:", np.__version__)
print("Pandas:", pd.__version__)

NumPy: 1.26.4
Pandas: 2.2.2


In [7]:
import sklearn
import osmnx as ox
import networkx as nx
import folium

print("OSMnx:", ox.__version__)

print("All libraries working!")
print("OSMnx:", osmnx.__version__)

OSMnx: 2.1.1
All libraries working!
OSMnx: 2.1.1


In [8]:
place = "Indiranagar, Bengaluru, India"

G = ox.graph_from_place(
    place,
    network_type="drive"
)

print("Road network downloaded successfully!")
print("Nodes:", len(G.nodes))
print("Edges:", len(G.edges))

Road network downloaded successfully!
Nodes: 322
Edges: 838


In [10]:
nodes = list(G.nodes)

source = nodes[0]
target = nodes[-1]

print("Source:", source)
print("Target:", target)
distance_route = nx.shortest_path(
    G,
    source=source,
    target=target,
    weight="length"
)

print("Shortest route found!")
print("Number of nodes:", len(distance_route))

Source: 253179982
Target: 13849804075
Shortest route found!
Number of nodes: 10


In [11]:
distance = nx.path_weight(
    G,
    distance_route,
    weight="length"
)

print(
    "Shortest route distance:",
    round(distance / 1000, 2),
    "km"
)

Shortest route distance: 0.58 km


In [12]:
traffic_factor = {
    "Low Traffic": 1.0,
    "Moderate Traffic": 1.5,
    "High Traffic": 2.0
}

# For this simple assignment, use the current overall traffic level
current_traffic = "High Traffic"

factor = traffic_factor[current_traffic]

print("Traffic condition:", current_traffic)
print("Traffic factor:", factor)

Traffic condition: High Traffic
Traffic factor: 2.0


In [13]:
for u, v, key, data in G.edges(
    keys=True,
    data=True
):
    
    length = data.get("length", 0)
    
    data["traffic_time"] = length * factor

print("Traffic weights added successfully!")

Traffic weights added successfully!


In [14]:
traffic_route = nx.shortest_path(
    G,
    source=source,
    target=target,
    weight="traffic_time"
)

print("Traffic-aware route found!")
print("Number of nodes:", len(traffic_route))

Traffic-aware route found!
Number of nodes: 10


In [15]:
traffic_distance = nx.path_weight(
    G,
    traffic_route,
    weight="length"
)

traffic_time = nx.path_weight(
    G,
    traffic_route,
    weight="traffic_time"
)

print(
    "Traffic-aware route distance:",
    round(traffic_distance / 1000, 2),
    "km"
)

print(
    "Traffic-aware route time:",
    round(traffic_time, 2),
    "relative time units"
)

Traffic-aware route distance: 0.58 km
Traffic-aware route time: 1166.61 relative time units


In [16]:
import random

random.seed(42)

for u, v, key, data in G.edges(
    keys=True,
    data=True
):
    
    length = data.get("length", 0)

    # Simple simulated traffic condition for each road
    factor = random.choice([
        1.0,   # Low traffic
        1.5,   # Moderate traffic
        2.0    # High traffic
    ])

    data["traffic_time"] = length * factor

print("Traffic-aware weights added!")

Traffic-aware weights added!


In [17]:
traffic_route = nx.shortest_path(
    G,
    source=source,
    target=target,
    weight="traffic_time"
)

traffic_distance = nx.path_weight(
    G,
    traffic_route,
    weight="length"
)

traffic_cost = nx.path_weight(
    G,
    traffic_route,
    weight="traffic_time"
)

print(
    "Traffic-aware route distance:",
    round(traffic_distance / 1000, 2),
    "km"
)

print(
    "Traffic-aware route cost:",
    round(traffic_cost, 2)
)

Traffic-aware route distance: 0.58 km
Traffic-aware route cost: 880.72


In [18]:
print("ROUTE COMPARISON")
print("-------------------------")

print(
    "Static route:",
    round(distance / 1000, 2),
    "km"
)

print(
    "Traffic-aware route:",
    round(traffic_distance / 1000, 2),
    "km"
)

print(
    "Static route cost:",
    round(distance, 2)
)

print(
    "Traffic-aware cost:",
    round(traffic_cost, 2)
)

ROUTE COMPARISON
-------------------------
Static route: 0.58 km
Traffic-aware route: 0.58 km
Static route cost: 583.31
Traffic-aware cost: 880.72


In [1]:
source_lat = G.nodes[source]["y"]
source_lon = G.nodes[source]["x"]

target_lat = G.nodes[target]["y"]
target_lon = G.nodes[target]["x"]

m = folium.Map(
    location=[source_lat, source_lon],
    zoom_start=14
)

# Source marker
folium.Marker(
    [source_lat, source_lon],
    popup="Source"
).add_to(m)

# Destination marker
folium.Marker(
    [target_lat, target_lon],
    popup="Destination"
).add_to(m)

m
static_coordinates = [
    (
        G.nodes[node]["y"],
        G.nodes[node]["x"]
    )
    for node in distance_route
]

folium.PolyLine(
    result["normal_coordinates"],
    color="blue",
    weight=6,
    opacity=0.8,
    popup="Shortest Distance Route"
).add_to(m)

traffic_coordinates = [
    (
        G.nodes[node]["y"],
        G.nodes[node]["x"]
    )
    for node in traffic_route
]

folium.PolyLine(
    result["traffic_coordinates"],
    color="red",
    weight=6,
    opacity=0.8,
    popup="Traffic-Aware Route"
).add_to(m)

m

m.save("../route_map.html")

print("Map saved successfully!")

NameError: name 'G' is not defined